In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import warnings as wr
wr.filterwarnings('ignore')

In [2]:
df = pd.read_csv('data/07_flats_EDA.csv')
df.dropna(inplace=True)    # Dropping rows with missing values for simplicity (can be handled better with imputation if needed)
df['price_cr'] = df['price_cr']*1.7      # Adjusting price to current market value (approx. 70% increase from 2021 to 2026)

In [3]:
X = df.iloc[:,1:]
y = df.iloc[:,0]

In [4]:
num_features = X.select_dtypes(include=['int64', 'float64']).columns
cat_features = X.select_dtypes(include=['object']).columns

In [5]:
ct = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', StandardScaler())
    ]), num_features),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('oe', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), cat_features),
], remainder='passthrough')

X = ct.fit_transform(X)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
def evaluate_model(y, y_pred):
    mae = mean_absolute_error(y, y_pred)
    mse = mean_squared_error(y, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y, y_pred)
    return mae, rmse, r2

In [8]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'KNN Regressor': KNeighborsRegressor(),
    'Decision Tree': DecisionTreeRegressor(),
    'Random Forest': RandomForestRegressor(),
    'Gradient Boosting': GradientBoostingRegressor(),
    'Ada Boost': AdaBoostRegressor(),
    'SVM': SVR(kernel='rbf', C=70, gamma=0.06, epsilon=.14),
    'XGBoost': XGBRegressor(),
    'CatBoost': CatBoostRegressor(verbose=False)
}

In [9]:
test_scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_mae, train_rmse, train_r2 = evaluate_model(y_train, y_train_pred)
    test_mae, test_rmse, test_r2 = evaluate_model(y_test, y_test_pred)

    test_scores[name] = test_r2

    print(f"{name}:")
    print(f"Training - MAE: {train_mae:.2f}, RMSE: {train_rmse:.2f}, R²: {train_r2:.4f}")
    print(f"Testing  - MAE: {test_mae:.2f}, RMSE: {test_rmse:.2f}, R²: {test_r2:.4f}\n")

best_model_name = max(test_scores, key=test_scores.get)
print(f"Best Model: {best_model_name} with R²: {test_scores[best_model_name]:.4f}")

Linear Regression:
Training - MAE: 227.10, RMSE: 1791.39, R²: 0.0127
Testing  - MAE: 358.52, RMSE: 3058.62, R²: -0.0015

Ridge Regression:
Training - MAE: 226.96, RMSE: 1791.39, R²: 0.0127
Testing  - MAE: 358.37, RMSE: 3058.60, R²: -0.0015

Lasso Regression:
Training - MAE: 222.64, RMSE: 1791.42, R²: 0.0127
Testing  - MAE: 353.97, RMSE: 3058.38, R²: -0.0014

KNN Regressor:
Training - MAE: 112.71, RMSE: 1589.79, R²: 0.2224
Testing  - MAE: 309.04, RMSE: 3225.81, R²: -0.1140

Decision Tree:
Training - MAE: 22.82, RMSE: 775.09, R²: 0.8152
Testing  - MAE: 600.61, RMSE: 5897.42, R²: -2.7234

Random Forest:
Training - MAE: 72.55, RMSE: 1029.22, R²: 0.6741
Testing  - MAE: 388.44, RMSE: 3621.90, R²: -0.4044

Gradient Boosting:
Training - MAE: 86.83, RMSE: 852.92, R²: 0.7762
Testing  - MAE: 413.81, RMSE: 4036.90, R²: -0.7447

Ada Boost:
Training - MAE: 69.26, RMSE: 806.28, R²: 0.8000
Testing  - MAE: 381.02, RMSE: 4290.73, R²: -0.9709

SVM:
Training - MAE: 70.78, RMSE: 1805.71, R²: -0.0031
Testin